## Crear insumo de programas clasificados en Portal TUNI

In [1]:
# Paquetes
from pathlib import Path

import unicodedata
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Rutas 
PROYECTO = Path(r"D:\Daniel_Pajita\1_Proyectos\Research\Educacion\educacion-universitaria-peru")
BASES_TUNI = Path(r"E:\BD\SUNEDU\TUNI\corte_16092026")
SALIDA = PROYECTO / "2_Datos" / "Correspondencias_entre_clasificaciones"

In [3]:
# Rangos de años y semestres
ANIOS_POSTULANTES = range(2022, 2026)
ANIOS_INGRESANTES = range(2020, 2026)
ANIOS_MATRICULADOS = range(2020, 2026)
ANIOS_EGRESADOS = range(2022, 2026)
SEMESTRES = ("I", "II")

COLUMNAS_PROGRAMA = [
    "codigo_siu_programa","codigo_grupo_1","nombre_grupo_1","codigo_grupo_3","nombre_grupo_3","nombre_programa","nivel_academico",
]

COLUMNAS_EXPORTACION = [
    "nivel_academico","codigo_grupo_1","nombre_grupo_1","codigo_grupo_3","nombre_grupo_3","nombre_programa",
]

In [4]:
# Funciones de lectura y preparación
def leer_columnas(path: Path, columnas: list[str]) -> pd.DataFrame:
    for encoding in ("utf-8-sig", "cp1252"):
        try:
            df = pd.read_csv(
                path,
                sep="|",
                usecols=lambda col: col.strip().lower() in columnas,
                dtype="string",
                keep_default_na=False,
                na_values=["<NULL>"],
                encoding=encoding,
            )

            df.columns = df.columns.str.strip().str.lower()
            exigir_columnas(df, columnas, path.name)

            for columna in df.columns:
                df[columna] = df[columna].str.strip().fillna("")

            return df[columnas]

        except UnicodeDecodeError:
            if encoding == "cp1252":
                raise


def exigir_columnas(
    df: pd.DataFrame, columnas: list[str], nombre: str
) -> None:
    faltantes = [col for col in columnas if col not in df.columns]
    if faltantes:
        raise KeyError(f"Faltan columnas en {nombre}: {faltantes}")


def filas_unicas(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates().reset_index(drop=True)


def apilar_unicos(bases: list[pd.DataFrame]) -> pd.DataFrame:
    return filas_unicas(pd.concat(bases, ignore_index=True))

In [5]:
# Programas de postulantes: primera, segunda y tercera opción
opciones_postulante = {}

for opcion in ["primera", "segunda", "tercera"]:
    opciones_postulante[opcion] = {
        f"codigo_siu_programa_{opcion}_opcion": "codigo_siu_programa",
        f"codigo_grupo_1_{opcion}_opcion": "codigo_grupo_1",
        f"nombre_grupo_1_{opcion}_o": "nombre_grupo_1",
        f"codigo_grupo_3_{opcion}_opcion": "codigo_grupo_3",
        f"nombre_grupo_3_{opcion}_o": "nombre_grupo_3",
        f"nombre_programa_{opcion}_opcion": "nombre_programa",
        "nivel_academico": "nivel_academico",
    }

columnas_origen = sorted({
    columna
    for mapeo in opciones_postulante.values()
    for columna in mapeo
})

bases_postulantes = []

for anio in ANIOS_POSTULANTES:
    fuente = leer_columnas(BASES_TUNI / f"1_postulantes/postulante_{anio}.txt",columnas_origen,)

    opciones_anio = []

    for mapeo in opciones_postulante.values():
        programas_opcion = (fuente[list(mapeo)].rename(columns=mapeo).loc[:, COLUMNAS_PROGRAMA])
        opciones_anio.append(filas_unicas(programas_opcion))

    bases_postulantes.append(apilar_unicos(opciones_anio))

postulantes = apilar_unicos(bases_postulantes)

print(f"Postulantes: {len(postulantes):,} combinaciones únicas")

Postulantes: 5,149 combinaciones únicas


In [6]:
# Programas de ingresantes, matriculados y egresados
def cargar_fuentes(paths: list[Path]) -> pd.DataFrame:
    bases = []

    for path in paths:
        base = leer_columnas(path, COLUMNAS_PROGRAMA)
        bases.append(filas_unicas(base))

    return apilar_unicos(bases)


ingresantes = cargar_fuentes([
    BASES_TUNI / f"2_ingresantes/ingresante_{anio}.txt"
    for anio in ANIOS_INGRESANTES
])

matriculados = cargar_fuentes([
    BASES_TUNI / f"3_matriculados/matriculado_{anio}_{semestre}.txt"
    for anio in ANIOS_MATRICULADOS
    for semestre in SEMESTRES
])

egresados = cargar_fuentes([
    BASES_TUNI / f"4_egresados/egresado_{anio}.txt"
    for anio in ANIOS_EGRESADOS
])

for nombre, base in [
    ("Ingresantes", ingresantes),
    ("Matriculados", matriculados),
    ("Egresados", egresados),
]:
    print(f"{nombre}: {len(base):,} combinaciones únicas")

Ingresantes: 5,277 combinaciones únicas
Matriculados: 5,679 combinaciones únicas
Egresados: 3,458 combinaciones únicas


In [7]:
## Programas de la oferta institucional
mapeo_oferta = {
    "programa_codigo": "codigo_siu_programa",
    "programa_codigo_grupo_1": "codigo_grupo_1",
    "programa_nombre_grupo_1": "nombre_grupo_1",
    "programa_codigo_grupo_3": "codigo_grupo_3",
    "programa_nombre_grupo_3": "nombre_grupo_3",
    "programa_nombre": "nombre_programa",
    "programa_nivel_academico": "nivel_academico",
}

oferta_path = BASES_TUNI / "7_oferta/institucional.txt"
oferta = leer_columnas(oferta_path, list(mapeo_oferta))
oferta = oferta.rename(columns=mapeo_oferta)[COLUMNAS_PROGRAMA]
oferta = oferta.loc[oferta["nivel_academico"].ne("")]
oferta = filas_unicas(oferta)

print(f"Oferta: {len(oferta):,} combinaciones únicas")

Oferta: 7,340 combinaciones únicas


In [ ]:
## Consolidación, limpieza y resolución de duplicados
programas = pd.concat([postulantes, ingresantes, matriculados, egresados, oferta],ignore_index=True,)
programas["nivel_academico"] = (programas["nivel_academico"].str.strip().replace({
        "Carrera Profesional": "Pregrado",
        "Segunda Especialidad": "Segunda especialidad",
    })
)

# Equivalente a las reglas de faltantes del do-file.
programas = programas.loc[programas["nombre_programa"].ne("")].copy()
programas["codigo_grupo_1"] = programas["codigo_grupo_1"].replace(".", "")
programas["codigo_grupo_3"] = programas["codigo_grupo_3"].replace({".": "", "X": ""})
programas["nombre_grupo_1"] = programas["nombre_grupo_1"].replace("Sin información", "")
programas["nombre_grupo_3"] = programas["nombre_grupo_3"].replace("Sin información", "")

# Stata usa destring. Int64 conserva faltantes y elimina ceros iniciales.
for columna in ["codigo_siu_programa", "codigo_grupo_1", "codigo_grupo_3"]:
    programas[columna] = pd.to_numeric(programas[columna].replace("", pd.NA), errors="raise").astype("Int64")
    
programas = programas.drop(columns="codigo_siu_programa").drop_duplicates()

# Replica gsort nombre_programa -codigo_grupo_1 -codigo_grupo_3 nivel_academico.
programas = programas.sort_values(
    ["nombre_programa", "codigo_grupo_1", "codigo_grupo_3", "nivel_academico"],
    ascending=[True, False, False, True],
    na_position="last",
    kind="mergesort",
)

# Conserva una clasificación por nombre de programa y nivel académico.
programas = programas.drop_duplicates(subset=["nivel_academico", "nombre_programa"], keep="first").reset_index(drop=True)
programas = programas[COLUMNAS_EXPORTACION]

print(f"Catálogo consolidado: {len(programas):,} programas")
programas.head()

Catálogo consolidado: 5,397 programas


,nivel_academico,codigo_grupo_1,nombre_grupo_1,codigo_grupo_3,nombre_grupo_3,nombre_programa
0,Maestría,8,"Agricultura, Silvicultura, Pesca y Veterinaria",831,Pesca,Acuicultura
1,Pregrado,8,"Agricultura, Silvicultura, Pesca y Veterinaria",831,Pesca,Acuicultura
2,Segunda especialidad,8,"Agricultura, Silvicultura, Pesca y Veterinaria",831,Pesca,Acuicultura y Gerenciamiento
3,Maestría,3,"Ciencias Sociales, Periodismo e Información",313,Psicología,Adicciones y Conductas Adictivas
4,Doctorado,4,Ciencias Administrativas y Derecho,413,Gestión y Administración,Administración


In [11]:
## Exportación de entrenamiento y predicción
entrenamiento = programas.loc[programas["codigo_grupo_1"].notna()].copy()
prediccion = programas.loc[programas["codigo_grupo_1"].isna()].copy()

path_catalogo = SALIDA / "BD_tuni_programas.txt"
path_entrenamiento = SALIDA / "BD_tuni_programas_entrenamiento.txt"
path_prediccion = SALIDA / "BD_tuni_programas_predicción.txt"

programas.to_csv(path_catalogo, sep="|", index=False, encoding="utf-8", na_rep="")
entrenamiento.to_csv(path_entrenamiento, sep="|", index=False, encoding="utf-8", na_rep="")
prediccion.to_csv(path_prediccion, sep="|", index=False, encoding="utf-8", na_rep="")

assert not programas.duplicated(["nivel_academico", "nombre_programa"]).any()
assert entrenamiento["codigo_grupo_1"].notna().all()
assert prediccion["codigo_grupo_1"].isna().all()

resumen = pd.DataFrame(
    {
        "base": ["catálogo", "entrenamiento", "predicción"],
        "filas": [len(programas), len(entrenamiento), len(prediccion)],
        "archivo": [path_catalogo.name, path_entrenamiento.name, path_prediccion.name],
    }
)
resumen

,base,filas,archivo
0,catálogo,5397,BD_tuni_programas.txt
1,entrenamiento,5122,BD_tuni_programas_entrenamiento.txt
2,predicción,275,BD_tuni_programas_predicción.txt
